# Offline final-voicing threshold sweep

Use the frozen 449-track paper cohort: 40 Bach10, 149 URMP and 260 CocoChorales tracks. Each track is freshly pitch-detected once with the live confidence/volume value gates disabled, then passed through the production HMM once. Ordinary pitch caches are never read or changed. Dedicated pre-gate arrays are saved so subsequent runs only mask and rescore.

The HMM uses the fixed production threshold **0.990** and volume floor **0.04 × p95(RMS)**. Only the final unvoiced-probability threshold varies; smaller values reject more frames. The grid is restricted to ≤0.990 because loosening beyond the HMM's tracking gate would require a different tracking experiment. This is not a sweep of the threshold inside the HMM. The 0.990 mask must exactly reproduce the production voicing controller on the freshly tracked path. Fresh detection can differ from saved paper caches; `baseline_comparison.csv` records those differences separately, so threshold effects are measured against the fresh 0.990 control.

Metrics pool frame counts on the standard 10 ms grid, with separate real-audio and CocoChorales summaries. These are exploratory tuning results, not held-out validation. Run with the `at-venv` kernel. Progress occupies one updating line; a failed track stops the run.


In [1]:
from pathlib import Path
import sys

REPO_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                 if (p / "app.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from benchmarks.modules.pitch.sweeps.FinalVoicingSweep import run_sweep


In [2]:
THRESHOLDS = (
    0.50, 0.70, 0.80, 0.85, 0.90, 0.925, 0.95, 0.96,
    0.97, 0.975, 0.98, 0.9825, 0.985, 0.9875, 0.990,
)
WORKERS = 8  # Parallel processes; one numerical thread per worker.
OUTPUT = REPO_ROOT / "benchmarks/results/pitch/final_voicing_449_2026-10-08"
BASELINE = (
    REPO_ROOT / "benchmarks/results/pitch/notebook_runs/paired_analysis"
    / "offline/4f615702b8c6/paired_rows.csv"
)


In [3]:
summary = run_sweep(
    thresholds=THRESHOLDS, workers=WORKERS, output=OUTPUT, baseline=BASELINE,
)


[449/449] complete; fresh control verified; 52 paper-baseline differences; /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/final_voicing_449_2026-10-08

In [4]:
display(summary.round(4))


,threshold,Group,Tracks,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
0,0.5000,All,449,84.6521,85.0994,86.9554,85.5444,3.1164
1,0.7000,All,449,89.0419,89.6050,90.2253,90.2771,4.6739
2,0.8000,All,449,90.9039,91.5136,91.5140,92.3473,5.8561
3,0.8500,All,449,91.7942,92.4265,92.0933,93.3584,6.6173
4,0.9000,All,449,92.5123,93.1614,92.5284,94.1828,7.4021
5,0.9250,All,449,93.0903,93.7528,92.8620,94.8563,8.1216
6,0.9500,All,449,93.5674,94.2438,93.1178,95.4239,8.8201
7,0.9600,All,449,93.9570,94.6433,93.3056,95.8919,9.5022
8,0.9700,All,449,94.2887,94.9830,93.4450,96.2949,10.1916
9,0.9750,All,449,94.4305,95.1279,93.5003,96.4676,10.5092
